# 03 · Thermal model setup

The model brings together building geometry, fabric properties, weather and household inputs.


In [ ]:
# Review copy: read saved results without running the research pipeline.
import os
if os.environ.get("ENERMAP_ENABLE_MODEL_RUN") != "1":
    raise RuntimeError("Review copy: source inputs and third-party engines are excluded. Read saved results or use the Streamlit app.")
import sys
from pathlib import Path
_repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path.insert(0, str(_repo))


In [ ]:
import matplotlib.pyplot as plt; plt.rcParams["savefig.dpi"] = 300   # publication resolution for every saved figure
import sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))
from config import *              # paths + modelling choices (edit config.py, not the notebooks)
import pbc_helpers as H
H.setup_plots()

import ukubem
from ukubem.engine import PBE, run_building, annual_summary
from ukubem.geometry import UKGeometryAssumptions, build_bui, rdsap_default_window_area
from ukubem.schedules import sample_household, dwelling_schedule, deterministic_schedule
from ukubem.systems import delivered_energy, SYSTEM_DEFAULTS
from ukubem.runner import simulate_rows, simulate_stock, aggregate_lsoa, mape
pybui = PBE.load(PBE_SRC)
UBEM_OUT = CURRENT_OUT / "method_examples"; UBEM_OUT.mkdir(exist_ok=True)
FIG_UBEM = CURRENT_OUT / "figures"; FIG_UBEM.mkdir(exist_ok=True)
print(f"pyBuildingEnergy from {PBE_SRC}   |   cores: {N_JOBS}")

## 1 · One dwelling, step by step

In [ ]:
inputs = pd.read_parquet(EXPORT_DIR / "dwelling_inputs.parquet")
table = pd.read_parquet(TABLE_NUMERIC)
labels = pd.read_parquet(TABLE_LABELS)
A = UKGeometryAssumptions()
print(f"{len(inputs):,} dwellings with archetype-enriched inputs; EPW = {EPW.name}")

did = inputs[(inputs.accommodation_type == "Semi-detached") & (inputs.age_band_resolved == "D")].index[0]
r = inputs.loc[did]
bui, derived = build_bui(r, A)
print(f"dwelling {did}: {r.accommodation_type}, {r.built_form}, band {r.age_band_resolved}, {r.floor_area_final:.0f} m², "
      f"{r.wall_construction_resolved} ({r.wall_insulation}), {r.glazing_type} glazing, {r.heat_system}")
pd.Series({k: (round(v, 3) if isinstance(v, (int, float, np.floating)) and not isinstance(v, bool) else v) for k, v in derived.items()})

In [ ]:
pd.DataFrame(bui["building_surface"])[["name", "type", "area", "u_value", "thermal_capacity", "orientation"]].round(2)

In [ ]:
t0 = time.time()
hourly, annual, secs = run_building(bui, EPW, PBE_SRC, use_weather_cache=False)
hourly2, annual2, secs2 = run_building(bui, EPW, PBE_SRC, use_weather_cache=True)
res = annual_summary(hourly, annual, derived["A_floor"])
print(f"first run (computes the weather frame) {secs:.1f} s, second run (cached) {secs2:.1f} s; identical: {np.isclose(hourly['Q_HC'].sum(), hourly2['Q_HC'].sum())}")
print(f"space-heating need {res['Q_H_kWh']:,.0f} kWh/yr = {res['Q_H_kWh_m2']:.0f} kWh/m²")
pd.Series(res).round(1)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
wk = hourly.loc["2009-01-12":"2009-01-18"]
axes[0].plot(wk.index, wk["T_op"], label="operative T"); axes[0].plot(wk.index, wk["T_ext"], label="outdoor T", alpha=0.7)
axes[0].set_ylabel("°C"); axes[0].legend(); axes[0].set_title(f"dwelling {did}: a January week, two-period timer regime (19.5 / 16 °C)")
axes[1].fill_between(wk.index, 0, wk["Q_HC"] / 1000, step="mid", alpha=0.6); axes[1].set_ylabel("heating kW")
plt.tight_layout(); fig.savefig(FIG_UBEM / "03_one_dwelling_week.png", bbox_inches="tight"); plt.show()

## 2 · Same dwelling with a stochastic household (CREST/CHAP regime)

In [ ]:
rng = np.random.default_rng(int(did))
hh = sample_household(r.accommodation_type, r.floor_area_final, r.tenure, rng)
sch = dwelling_schedule(str(did), hh, r.floor_area_final, year=2021, seed=int(did), timestep_minutes=60)
bui_s, der_s = build_bui(r, A, schedule=sch)
hourly_s, annual_s, secs_s = run_building(bui_s, EPW, PBE_SRC, external_internal_gains_w=sch["gains_w"])
res_s = annual_summary(hourly_s, annual_s, der_s["A_floor"])
print(f"household {hh} ({sch['people']} people), EFUS pattern {sch['heating_pattern']}, thermostat {sch['setpoint_c']:.1f} °C, "
      f"season start month {sch['season_start_month']} for {sch['season_length_months']} months, heating available {100*sch['heating_share_of_hours']:.0f} % of hours")
print(f"space-heating need {res_s['Q_H_kWh']:,.0f} kWh/yr = {res_s['Q_H_kWh_m2']:.0f} kWh/m² (deterministic regime: {res['Q_H_kWh_m2']:.0f});  "
      f"DHW {sch['dhw_kWh_year']:,.0f} kWh, appliances+lighting {sch['elec_kWh']:,.0f} kWh")
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
wk = hourly_s.loc["2009-01-12":"2009-01-18"]; idx = pd.date_range("2021-01-11", periods=24*7, freq="h")
pos = (idx.dayofyear - 1) * 24 + idx.hour
axes[0].plot(wk.index, wk["T_op"], label="operative T"); axes[0].plot(wk.index, wk["T_ext"], label="outdoor T", alpha=0.7); axes[0].legend(); axes[0].set_ylabel("°C")
axes[1].fill_between(wk.index, 0, wk["Q_HC"] / 1000, step="mid", alpha=0.6); axes[1].set_ylabel("heating kW")
axes[2].step(wk.index, sch["heating_on"][pos], where="mid", label="heating available"); axes[2].plot(wk.index, sch["gains_w"][pos] / 1000, label="internal gains kW"); axes[2].legend(); axes[2].set_ylabel("")
axes[0].set_title(f"dwelling {did} with a {hh} household: {sch['heating_pattern']}")
plt.tight_layout(); fig.savefig(FIG_UBEM / "03_one_dwelling_week_stochastic.png", bbox_inches="tight"); plt.show()

## 3 · One dwelling per construction sub-archetype (the 38 medoids), both regimes, all cores

This is the shape of every later run: a table of dwellings in, one summary row per dwelling
out, `joblib` across the 32 cores, each worker keeping its own weather cache.

In [ ]:
from joblib import Parallel, delayed
reps = json.load(open(EXPORT_DIR / "cluster_representatives.json"))
medoids = [int(v["medoid"]["dwelling_id"]) for v in reps.values()]
rows = inputs.loc[medoids]
t0 = time.time()
chunks = [rows.iloc[i:i + 2] for i in range(0, len(rows), 2)]
det = sum(Parallel(n_jobs=N_JOBS)(delayed(simulate_rows)(c, str(EPW), str(PBE_SRC), A, False, SEED, 2021, None) for c in chunks), [])
sto = sum(Parallel(n_jobs=N_JOBS)(delayed(simulate_rows)(c, str(EPW), str(PBE_SRC), A, True, SEED, 2021, None) for c in chunks), [])
det, sto = pd.DataFrame(det).set_index("dwelling_id"), pd.DataFrame(sto).set_index("dwelling_id")
print(f"{2*len(rows)} simulations in {(time.time()-t0)/60:.1f} min on {N_JOBS} cores  ({det['seconds'].mean():.1f} s each inside the worker)")
cmp = pd.DataFrame({"archetype": inputs.loc[medoids, "construction_sa"].values,
                    "type": inputs.loc[medoids, "accommodation_type"].values, "age": inputs.loc[medoids, "age_band_resolved"].values,
                    "A_m2": det["A_floor"].values, "HTC_W_m2K": det["HTC_per_m2"].values,
                    "ISO52016_timer_regime": det["Q_H_kWh_m2"].values, "ISO52016_CHAP_household": sto["Q_H_kWh_m2"].values,
                    "household": sto["household_archetype"].values, "pattern": sto["heating_pattern"].values,
                    "T_op_winter_CHAP": sto["T_op_heating_season_C"].values}, index=medoids)
cmp.to_csv(UBEM_OUT / "03_medoids_two_regimes.csv")
cmp.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lim = cmp[["ISO52016_timer_regime", "ISO52016_CHAP_household"]].max().max() * 1.05
axes[0].scatter(cmp["ISO52016_timer_regime"], cmp["ISO52016_CHAP_household"], s=28, alpha=0.8)
for i, row in cmp.iterrows():
    axes[0].annotate(row["archetype"].replace("C.", ""), (row["ISO52016_timer_regime"], row["ISO52016_CHAP_household"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
axes[0].plot([0, lim], [0, lim], "k--", lw=1); axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel("two-period timer regime 19.5/16 °C  [kWh/m²/yr]"); axes[0].set_ylabel("one CHAP household draw  [kWh/m²/yr]"); axes[0].set_title("the same medoids under the two regimes (uncalibrated fabric)", fontsize=9)
order_ = cmp.sort_values("ISO52016_timer_regime").index
axes[1].barh(range(len(order_)), cmp.loc[order_, "ISO52016_timer_regime"], color="grey", alpha=0.6, label="timer regime")
axes[1].barh(range(len(order_)), cmp.loc[order_, "ISO52016_CHAP_household"], color="tab:red", alpha=0.6, height=0.5, label="CHAP household")
axes[1].set_yticks(range(len(order_))); axes[1].set_yticklabels(cmp.loc[order_, "archetype"], fontsize=6); axes[1].set_xlabel("kWh/m²/yr"); axes[1].legend(); axes[1].set_title("intensity by sub-archetype medoid", fontsize=9)
plt.tight_layout(); fig.savefig(FIG_UBEM / "03_medoids_two_regimes.png", bbox_inches="tight"); plt.show()
r1 = np.corrcoef(cmp["ISO52016_timer_regime"], cmp["ISO52016_CHAP_household"])[0, 1]
print(f"correlation between the regimes across medoids: {r1:.2f}; median ratio household/timer = {(cmp['ISO52016_CHAP_household']/cmp['ISO52016_timer_regime']).median():.2f}")
print("Uncalibrated fabric with either regime is the starting point; notebook 06 compares the uncalibrated results against the LSOA meters without fitting parameters.")

**Next:** `04_household_schedules.ipynb` — the stochastic household layer in detail.